# Homelab smoke test

Checks that this notebook can drive the Spark cluster (`spark://spark-master:7077`),
that executors run the same Python as the driver, that Delta `MERGE INTO` works,
and (optionally) that the external SQL Server is reachable over JDBC.

Cluster config comes from `/opt/spark-conf/spark-defaults.conf` (mounted from `conf/spark/`).

In [ ]:
import sys
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("jupyter-smoke-test").getOrCreate()
sc = spark.sparkContext
print("Spark:", spark.version, "| master:", sc.master)
print("Driver Python:", sys.version.split()[0])

In [ ]:
# Runs Python on the executors: shows which worker ran it and with which Python.
import socket

def whoami(_):
    import sys, socket
    return (socket.gethostname(), sys.version.split()[0])

sorted(set(sc.parallelize(range(16), 8).map(whoami).collect()))

In [ ]:
# Delta Lake: create a target table, then upsert with MERGE INTO.
target_path = "/opt/spark-data/output/customers_delta"

spark.createDataFrame(
    [("C001", "Alice", "North"), ("C002", "Bob", "South")],
    "customer_id string, name string, region string",
).write.format("delta").mode("overwrite").save(target_path)

spark.createDataFrame(
    [("C002", "Bob", "East"), ("C003", "Chidi", "West")],
    "customer_id string, name string, region string",
).createOrReplaceTempView("customer_updates")

spark.sql(f"""
    MERGE INTO delta.`{target_path}` AS t
    USING customer_updates AS s
    ON t.customer_id = s.customer_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""").show()

spark.read.format("delta").load(target_path).orderBy("customer_id").show()

In [ ]:
from delta.tables import DeltaTable

DeltaTable.forPath(spark, target_path).history().select("version", "operation", "operationMetrics").show(truncate=False)

## Optional: external SQL Server over JDBC

Needs `MSSQL_*` values in `.env`, and SQL Server must accept TCP/IP connections with SQL authentication on port 1433.

In [ ]:
import os

jdbc_url = (
    f"jdbc:sqlserver://{os.environ['MSSQL_HOST']}:{os.environ['MSSQL_PORT']};"
    f"databaseName={os.environ['MSSQL_DB']};encrypt=true;trustServerCertificate=true"
)

(spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT @@SERVERNAME AS server_name, @@VERSION AS version")
    .option("user", os.environ["MSSQL_USER"])
    .option("password", os.environ["MSSQL_PASSWORD"])
    .load()
    .show(truncate=80))

In [ ]:
spark.stop()